In [3]:
import sys
import warnings
import tempfile
import numpy as np
from pathlib import Path

from Bio.PDB import PDBParser, NeighborSearch, Select
from Bio.PDB.PDBIO import PDBIO
from Bio.SeqUtils import seq1
from Bio import pairwise2
from Bio.PDB.Superimposer import Superimposer

import MDAnalysis as mda

from loguru import logger

warnings.filterwarnings("ignore", category=UserWarning, module="MDAnalysis.core.universe")

logger.remove()
logger.add(sys.stdout, format="{message}", level="INFO")

class LigandSelect(Select):
    def __init__(self, ligand_residues, interacting_chains):
        self.ligand_residues = ligand_residues
        self.interacting_chains = interacting_chains
        self.ligand_ids = set()
        for residue in ligand_residues:
            self.ligand_ids.add((residue.get_parent().id, residue.id))

    def accept_chain(self, chain):
        accept = chain.id in self.interacting_chains or any(chain.id == residue.get_parent().id for residue in self.ligand_residues)
        return accept

    def accept_residue(self, residue):
        chain_id = residue.get_parent().id
        if (chain_id, residue.id) in self.ligand_ids:
            return True
        elif residue.id[0] == ' ':
            return True
        else:
            return False
def calculate_aligned_rmsd(structure1_file, structure2_file):
    logger.info(f"Вычисление RMSD между {structure1_file} и {structure2_file}")
    
    parser = PDBParser(QUIET=True)
    structure1 = parser.get_structure('struct1', structure1_file)
    structure2 = parser.get_structure('struct2', structure2_file)
    
    # Извлекаем остатки белка
    residues1 = [res for res in structure1.get_residues() if res.id[0] == ' ']
    residues2 = [res for res in structure2.get_residues() if res.id[0] == ' ']
    
    # Получаем последовательности
    seq1_str = ''.join([seq1(res.get_resname()) for res in residues1])
    seq2_str = ''.join([seq1(res.get_resname()) for res in residues2])

    if not seq1_str or not seq2_str:
        logger.warning("Одна или обе последовательности пусты, невозможно выполнить выравнивание.")
        return np.inf
    
    # Выравниваем последовательности
    alignments = pairwise2.align.globalxx(seq1_str, seq2_str)

    if not alignments:
        logger.warning("Не удалось выполнить выравнивание последовательностей.")
        return np.inf

    alignment = alignments[0]
    
    idx1 = 0
    idx2 = 0
    atom_pairs = []
    for a, b in zip(alignment.seqA, alignment.seqB):
        if a != '-' and b != '-':
            res1 = residues1[idx1]
            res2 = residues2[idx2]
            for atom_name in ['N', 'CA', 'C', 'O']:
                if atom_name in res1 and atom_name in res2:
                    atom1 = res1[atom_name]
                    atom2 = res2[atom_name]
                    atom_pairs.append((atom1, atom2))
        if a != '-':
            idx1 += 1
        if b != '-':
            idx2 += 1
    
    if len(atom_pairs) == 0:
        logger.warning("Нет общих атомов для сравнения.")
        return np.inf
    
    # Вычисляем центр кармана на основе лигандов в первой структуре
    ligand_atoms1 = [atom for atom in structure1.get_atoms() if atom.get_parent().id[0] != ' ']
    if not ligand_atoms1:
        logger.warning("Нет лигандов в первой структуре для определения центра кармана.")
        return np.inf
    ligand_coords1 = np.array([atom.get_coord() for atom in ligand_atoms1])
    pocket_center = np.mean(ligand_coords1, axis=0)
    
    # Итеративное выравнивание с отбрасыванием выбросов
    sup = Superimposer()
    max_iterations = 10
    tolerance = 1e-4
    prev_rms = None
    for iteration in range(max_iterations):
        coords1 = np.array([atom1.get_coord() for atom1, atom2 in atom_pairs])
        coords2 = np.array([atom2.get_coord() for atom1, atom2 in atom_pairs])
        
        sup.set_atoms([atom1 for atom1, atom2 in atom_pairs], [atom2 for atom1, atom2 in atom_pairs])
        rmsd_value = sup.rms
        sup.apply([atom2 for atom1, atom2 in atom_pairs])
        
        # Вычисляем отклонения
        deviations = np.sqrt(np.sum((coords1 - coords2) ** 2, axis=1))
        rmsd_current = np.sqrt(np.mean(deviations ** 2))
        
        if prev_rms is not None and abs(prev_rms - rmsd_current) < tolerance:
            break
        prev_rms = rmsd_current
        
        # Отбрасываем выбросы, но сохраняем атомы в пределах 8 Å от центра кармана
        threshold = np.median(deviations) + 2 * np.std(deviations)
        new_atom_pairs = []
        for i, (atom1, atom2) in enumerate(atom_pairs):
            distance_from_pocket = np.linalg.norm(atom1.get_coord() - pocket_center)
            if deviations[i] < threshold or distance_from_pocket < 8.0:
                new_atom_pairs.append((atom1, atom2))
        atom_pairs = new_atom_pairs
        if len(atom_pairs) < 3:
            logger.warning("Слишком мало атомов после отбрасывания выбросов.")
            return np.inf
        
    logger.info(f"Значение RMSD между {structure1_file} и {structure2_file}: {rmsd_current:.3f}")
    return rmsd_current


def get_interacting_chains(ligand_atoms, protein_atoms, distance):
    ns = NeighborSearch(protein_atoms)
    interacting_atoms = []
    for atom in ligand_atoms:
        nearby_atoms = ns.search(atom.coord, distance, level='A')
        interacting_atoms.extend(nearby_atoms)
    interacting_chains = set(atom.get_parent().get_parent().id for atom in interacting_atoms)
    return interacting_chains

def find_close_ligands(ligand_atoms, all_ligand_atoms, distance):
    ns = NeighborSearch(all_ligand_atoms)
    close_residues = set()
    for atom in ligand_atoms:
        nearby_atoms = ns.search(atom.coord, distance, level='A')
        for nearby_atom in nearby_atoms:
            residue = nearby_atom.get_parent()
            if residue not in ligand_atoms and residue.id[0] != ' ':
                close_residues.add(residue)
    return list(close_residues)

def save_structure_to_tempfile(structure, select):
    io = PDBIO()
    temp_file = tempfile.NamedTemporaryFile(suffix=".pdb", delete=False)
    io.set_structure(structure)
    io.save(temp_file.name, select=select)
    temp_file.close()
    return temp_file.name

def process_ligands(structure, interact_distance=4.5, ligand_ligand_distance=3.0):
    logger.info(f"Обработка лигандов с расстоянием взаимодействия {interact_distance} и расстоянием лиганд-лиганд {ligand_ligand_distance}")
    all_ligand_atoms = [atom for atom in structure.get_atoms() if atom.get_parent().id[0] != ' ']
    protein_atoms = [atom for atom in structure.get_atoms() if atom.get_parent().id[0] == ' ']

    ligand_residues = [residue for residue in structure.get_residues() if residue.id[0] != ' ']
    processed_ligands = set()

    ligand_groups = []

    for ligand in ligand_residues:
        ligand_id = (ligand.get_resname(), ligand.get_parent().id, ligand.id[1])
        if ligand_id in processed_ligands:
            continue
        ligand_atoms = list(ligand.get_atoms())
        interacting_chains = get_interacting_chains(ligand_atoms, protein_atoms, interact_distance)
        if not interacting_chains:
            continue
        close_ligands = find_close_ligands(ligand_atoms, all_ligand_atoms, ligand_ligand_distance)
        all_ligands_in_group = [ligand] + close_ligands
        for close_ligand in close_ligands:
            processed_ligands.add((close_ligand.get_resname(), close_ligand.get_parent().id, close_ligand.id[1]))
        processed_ligands.add(ligand_id)
        ligand_groups.append({
            'ligands': all_ligands_in_group,
            'interacting_chains': interacting_chains
        })
        ligand_names = ", ".join([f"{lig.get_resname()} in chain {lig.get_parent().id}" for lig in all_ligands_in_group])
        logger.info(f"Найден карман с лигандами: {ligand_names} взаимодействует с цепями {', '.join(interacting_chains)}")

    logger.info(f"Всего найдено лигандных карманов: {len(ligand_groups)}")
    return ligand_groups

def save_pocket_structure(structure, pocket_info, output_dir, saved_structures, original_lines, input_file_path, interact_distance=4.5, rmsd_threshold=2.0):
    ligands = pocket_info['ligands']
    interacting_chains = pocket_info['interacting_chains']
    input_filename = Path(input_file_path).stem

    io = PDBIO()
    io.set_structure(structure)

    select = LigandSelect(ligands, interacting_chains)
    ligand_names = "_".join(sorted(set([ligand.get_resname() for ligand in ligands])))
    chains_str = "_".join(sorted(interacting_chains))
    output_file = output_dir / f"{input_filename}_{ligand_names}_chains_{chains_str}.pdb"

    temp_structure_file = save_structure_to_tempfile(structure, select)

    similar_found = False
    for saved in saved_structures:
        rmsd_value = calculate_aligned_rmsd(saved['temp_file'], temp_structure_file)
        logger.info(f"Значение RMSD: {rmsd_value:.3f}")

        if rmsd_value < rmsd_threshold:
            logger.info(f"Найдена похожая структура для {output_file.name} (RMSD: {rmsd_value:.3f} < {rmsd_threshold}), не сохраняется")
            similar_found = True
            break
    if similar_found:
        return

    with open(output_file, 'w') as f_out:
        f_out.writelines(original_lines)
        io.save(f_out, select=select)

    saved_structures.append({
        'output_file': output_file,
        'temp_file': temp_structure_file
    })
    logger.info(f"Сохранена новая структура: {output_file}")

def load_original_lines(file_path):
    with open(file_path, 'r') as f:
        lines = [line for line in f if not line.startswith(("ATOM", "HETATM", "END", "MASTER", "CONECT"))]
    return lines

def process_pdb_file(input_file_path, output_dir_path):
    interact_distance = 4.5
    ligand_ligand_distance = 3.0
    rmsd_threshold = 2.0

    output_dir = Path(output_dir_path)
    output_dir.mkdir(exist_ok=True)

    original_lines = load_original_lines(input_file_path)

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pdb_structure', input_file_path)

    ligand_groups = process_ligands(structure, interact_distance, ligand_ligand_distance)

    saved_structures = []

    for pocket_info in ligand_groups:
        save_pocket_structure(structure, pocket_info, output_dir, saved_structures, original_lines, input_file_path, interact_distance, rmsd_threshold)

if __name__ == "__main__":
    input_file_path = '/mnt/ligandpro/db/LPCE/processed/pdb8ceu.pdb'
    output_dir_path = 'separated_complexes_test'

    process_pdb_file(input_file_path, output_dir_path)


Обработка лигандов с расстоянием взаимодействия 4.5 и расстоянием лиганд-лиганд 3.0
Найден карман с лигандами: 1MG in chain a, 1MG in chain a взаимодействует с цепями a, d
Найден карман с лигандами: 2MG in chain a, 2MG in chain a взаимодействует с цепями a, C
Найден карман с лигандами: 3TD in chain a, 3TD in chain a взаимодействует с цепями a
Найден карман с лигандами: 2MG in chain a, 2MG in chain a взаимодействует с цепями a, e
Найден карман с лигандами: H2U in chain a, H2U in chain a взаимодействует с цепями a
Найден карман с лигандами: OMC in chain a, OMC in chain a взаимодействует с цепями a
Найден карман с лигандами: 2MA in chain a, 2MA in chain a взаимодействует с цепями a
Найден карман с лигандами: G34 in chain a, G34 in chain a взаимодействует с цепями a
Найден карман с лигандами: MEQ in chain d, MEQ in chain d взаимодействует с цепями a, d
Найден карман с лигандами: 4D4 in chain l, 4D4 in chain l, MS6 in chain l взаимодействует с цепями a, l
Найден карман с лигандами: KBE in c

KeyboardInterrupt: 

In [113]:
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed
from Bio.PDB import PDBParser

def find_pdb_files_with_multiple_ligands(folder_path):
    def count_unique_ligands(pdb_file):
        parser = PDBParser(QUIET=True)
        try:
            structure = parser.get_structure('', pdb_file)
            ligands = set()
            for model in structure:
                for chain in model:
                    for residue in chain:
                        if residue.id[0] == ' ':  # Пропуск стандартных аминокислот
                            continue
                        ligands.add(residue.get_resname())
            return pdb_file, len(ligands)
        except Exception:
            return pdb_file, 0

    pdb_files = list(Path(folder_path).rglob("*.pdb"))
    results = Parallel(n_jobs=-1)(
        delayed(count_unique_ligands)(pdb_file) for pdb_file in tqdm(pdb_files)
    )
    filtered_results = [(file, count) for file, count in results if count >= 2]
    sorted_results = sorted(filtered_results, key=lambda x: x[1], reverse=True)
    
    return sorted_results

# Пример использования
if __name__ == "__main__":
    folder_path = '/mnt/ligandpro/db/LPCE/processed'
    results = find_pdb_files_with_multiple_ligands(folder_path)
    for file, count in results:
        print(f"{file}: {count} лиганда(ов)")


  7%|▋         | 4480/60990 [00:04<01:02, 908.10it/s] 

100%|██████████| 60990/60990 [01:24<00:00, 725.86it/s]


/mnt/ligandpro/db/LPCE/processed/pdb8ceu.pdb: 14 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb6xz7.pdb: 14 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb8g6y.pdb: 12 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb6jv7.pdb: 12 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb2rqo.pdb: 12 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb8cep.pdb: 12 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb7qq3.pdb: 12 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb8g6x.pdb: 11 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb5xpk.pdb: 11 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb8cgd.pdb: 11 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb8rw1.pdb: 10 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb5goj.pdb: 10 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb4oik.pdb: 10 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb5jby.pdb: 10 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb5goi.pdb: 10 лиганда(ов)
/mnt/ligandpro/db/LPCE/processed/pdb5go8.pdb: 10 лиганда(ов)
/mnt/ligandpro/db/LPCE/p

In [5]:
!cp /mnt/ligandpro/db/LPCE/processed/pdb4i4t.pdb .


In [9]:
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed

def calculate_hetatm_percentage_and_first_ligand(pdb_file_path):
    atom_count = 0
    hetatm_count = 0
    first_hetatm_ligand = None
    
    with open(pdb_file_path, 'r') as file:
        for line in file:
            if line.startswith('ATOM'):
                atom_count += 1
            elif line.startswith('HETATM'):
                hetatm_count += 1
                if first_hetatm_ligand is None:
                    # Извлечение типа HETATM лиганда из позиции 12-16
                    first_hetatm_ligand = line[12:16].strip()
    
    total_count = atom_count + hetatm_count
    if total_count == 0:
        return pdb_file_path.name, 0, None  # Если нет записей ATOM или HETATM
    else:
        hetatm_percentage = (hetatm_count / total_count) * 100
        return pdb_file_path.name, hetatm_percentage, first_hetatm_ligand

def rank_pdb_files_by_hetatm(folder_path, high_threshold=50, low_threshold=10):
    pdb_files = list(Path(folder_path).rglob("*.pdb"))
    
    # Параллельная обработка файлов
    results = Parallel(n_jobs=-1)(
        delayed(calculate_hetatm_percentage_and_first_ligand)(pdb_file) for pdb_file in tqdm(pdb_files)
    )
    
    # Сортировка по убыванию процента HETATM
    sorted_results = sorted(results, key=lambda x: x[1], reverse=True)
    
    unique_ligands = set()

    print(f"PDB-файлы с HETATM > {high_threshold}%:\n")
    for pdb_name, hetatm_percentage, ligand_type in sorted_results:
        if hetatm_percentage > high_threshold:
            print(f"{pdb_name}: {hetatm_percentage:.2f}% HETATM")
        if ligand_type:
            unique_ligands.add(ligand_type)
    
    print(f"\nPDB-файлы с HETATM < {low_threshold}%:\n")
    for pdb_name, hetatm_percentage, ligand_type in sorted_results:
        if hetatm_percentage < low_threshold and ligand_type:
            print(f"{pdb_name}: {hetatm_percentage:.2f}% HETATM ({ligand_type})")
            unique_ligands.add(ligand_type)

    # Сохраняем уникальные лиганды в формате JSON
    ligands_json = {ligand: "" for ligand in sorted(unique_ligands)}
    
    return ligands_json

# Пример использования
folder_path = Path('/mnt/ligandpro/db/LPCE/processed')
json_ligands = rank_pdb_files_by_hetatm(folder_path, high_threshold=50, low_threshold=10)


100%|██████████| 60990/60990 [00:15<00:00, 3855.58it/s]


PDB-файлы с HETATM > 50%:

pdb7qdi.pdb: 98.34% HETATM
pdb3mbs.pdb: 98.17% HETATM
pdb3c1p.pdb: 98.09% HETATM
pdb1xj9.pdb: 97.67% HETATM
pdb6phm.pdb: 97.49% HETATM
pdb6phq.pdb: 97.19% HETATM
pdb2mh5.pdb: 96.04% HETATM
pdb2mjq.pdb: 95.52% HETATM
pdb2mjr.pdb: 95.52% HETATM
pdb2mjt.pdb: 95.38% HETATM
pdb3try.pdb: 95.35% HETATM
pdb2mjs.pdb: 95.34% HETATM
pdb1a7z.pdb: 95.12% HETATM
pdb2kql.pdb: 94.87% HETATM
pdb4glu.pdb: 94.78% HETATM
pdb6phn.pdb: 93.57% HETATM
pdb5f1w.pdb: 92.54% HETATM
pdb2q33.pdb: 92.06% HETATM
pdb4ttk.pdb: 90.91% HETATM
pdb3mbu.pdb: 89.19% HETATM
pdb1c4b.pdb: 85.12% HETATM
pdb3go0.pdb: 85.00% HETATM
pdb1fvm.pdb: 84.91% HETATM
pdb2jue.pdb: 82.50% HETATM
pdb1hhu.pdb: 82.50% HETATM
pdb3ipn.pdb: 82.00% HETATM
pdb1hhy.pdb: 81.61% HETATM
pdb1hzs.pdb: 81.25% HETATM
pdb1qpy.pdb: 80.39% HETATM
pdb1ym8.pdb: 80.09% HETATM
pdb1wzb.pdb: 80.00% HETATM
pdb1rru.pdb: 80.00% HETATM
pdb1bfw.pdb: 78.29% HETATM
pdb8g82.pdb: 77.78% HETATM
pdb1qfi.pdb: 74.90% HETATM
pdb1aa5.pdb: 74.65% HETATM
p

In [10]:
json_ligands

{'AC': '',
 'AG': '',
 'AM': '',
 'AR': '',
 'AS': '',
 'AS1': '',
 'AU': '',
 'B': '',
 'B01': '',
 'B02': '',
 'B03': '',
 'B1': '',
 'B10': '',
 'B13': '',
 'B18': '',
 'B33': '',
 'B63': '',
 'B7': '',
 'BAI': '',
 'BD': '',
 'BE': '',
 'BN1': '',
 'BR': '',
 'BR01': '',
 'BR1': '',
 'BR2': '',
 'BR4': '',
 'BR6': '',
 'BR7': '',
 'C': '',
 "C'": '',
 'C0': '',
 'C01': '',
 'C02': '',
 'C03': '',
 'C04': '',
 'C05': '',
 'C06': '',
 'C07': '',
 'C08': '',
 'C09': '',
 'C1': '',
 "C1'": '',
 'C10': '',
 'C11': '',
 'C12': '',
 'C13': '',
 'C14': '',
 'C15': '',
 'C16': '',
 'C17': '',
 'C18': '',
 'C19': '',
 'C1A': '',
 'C1B': '',
 'C1C': '',
 'C1G': '',
 'C1K': '',
 'C1Q': '',
 'C1R': '',
 'C1W': '',
 'C2': '',
 "C2'": '',
 'C20': '',
 'C21': '',
 'C22': '',
 'C23': '',
 'C24': '',
 'C25': '',
 'C26': '',
 'C27': '',
 'C28': '',
 'C29': '',
 'C2A': '',
 'C2B': '',
 'C3': '',
 "C3'": '',
 'C30': '',
 'C31': '',
 'C32': '',
 'C33': '',
 'C34': '',
 'C35': '',
 'C36': '',
 'C37': '',

ЗАДАЧИ

доделать скрипт который бьёт на файлы


проверить структукры где много лигандов


проверить структкуры где нарушен баланс HETATM/ATM